In [91]:
import argparse
import sys
import re
from pathlib import Path
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Union
from openpyxl import load_workbook



In [92]:
def drop_bad_reads(df, times=[], images=[]):
    df = df.copy()
    
    if times and "Elapsed" in df.columns:
    # keep rows where Elapsed is NOT in the times list using Series.isin
        df = df[~df["Elapsed"].isin(times)]    
    if images:
        cols = df.columns.astype(str)
        cols_to_drop = []
        for col in cols:
            if ", Image" in col:
                m = re.search(r'Image\s*(\d+)$', col)
                if m:
                    try:
                        num = int(m.group(1))
                        if num in images:
                            cols_to_drop.append(col)
                    except ValueError:
                        pass
        if cols_to_drop:
            df = df.drop(columns=cols_to_drop)
           
    return df.reset_index(drop=True)

def make_average_column_for_well(df, num_reads_per_well = 9):
    df = df.copy()
    new_averages = {}
    cols = df.columns.astype(str)
    #only use numeric cols
    numeric_cols = cols[3:]
    #print(f"len {len(numeric_cols)}, reads used: {num_reads_per_well}, remainder: {len(numeric_cols) % num_reads_per_well}")
    
    if len(numeric_cols) % num_reads_per_well == 0: #make sure the number won't go out of range
        for i in range(0, len(numeric_cols), num_reads_per_well):
            cols_to_avg = [numeric_cols[i] for i in range(i,i+num_reads_per_well)]
            split_header = cols_to_avg[0].split(",")
            well_name = split_header[0]
            new_averages[well_name] = df[cols_to_avg].mean(axis=1)
        header_df = df[cols[:3]]
        
        new_df = pd.DataFrame(new_averages)
        return pd.concat([header_df,new_df], axis=1)
    else:
        return pd.DataFrame()



In [93]:
def make_average_excel_with_exclusions(
input,
output,
inplace=False,
input_sheet = "Reformatted",
output_sheet="Averages",
overwrite=False,
time_rows_to_drop = [0],
images_to_drop = [5],
display=False
):
    inp = Path(input)
    if not inp.exists():
        print(f"Input file does not exist: {inp}", file=sys.stderr)
        sys.exit(2)

    if inplace and output:
        print("Cannot use --inplace and --output together.", file=sys.stderr)
        sys.exit(2)

    if input_sheet is not None:
        # allow numeric index if user provides digits
        try:
            sheet_name = int(input_sheet)
        except ValueError:
            sheet_name = input_sheet
    else:
        sheet_name = "Sheet1"

    df = pd.read_excel(inp, sheet_name=sheet_name)
    
    # apply replacements
    drop_df = drop_bad_reads(df,time_rows_to_drop,images_to_drop)
    
    number_images_dropped = len(images_to_drop)
    increment = int(9-number_images_dropped)
    out_df = make_average_column_for_well(drop_df, increment)
    if display:
        display(df)
        display(drop_df)
        display(out_df)

    # determine output path
    if inplace:
        out_path = inp
    elif output:
        out_path = Path(output)
    else:
        out_path = inp.with_name(inp.stem + "_editded" + inp.suffix)

    if Path.exists(out_path):
        try:
            with pd.ExcelWriter(out_path, engine="openpyxl", mode="a") as writer:
                out_df.to_excel(writer, sheet_name=output_sheet)
        except ValueError:
            if overwrite:
                with pd.ExcelWriter(out_path, engine="openpyxl", mode="w") as writer:
                    out_df.to_excel(writer, sheet_name=output_sheet)
            else:
                ValueError("Overwrite is disabled and sheet exists")
    else:
        # write back to excel
        out_df.to_excel(out_path, sheet_name=output_sheet, index=False)
    print(f"Wrote replaced data to: {out_path}")






In [94]:
for i in range(1,8):
    plate = i
    input= f"/Users/allielas/Desktop/Active Projects/Quality control/Incucyte sheets/r{plate}.xlsx"
    output= f"/Users/allielas/Desktop/Active Projects/Quality control/Incucyte sheets/r{plate}_processed.xlsx"
    make_average_excel_with_exclusions(input,output)


/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


Wrote replaced data to: /Users/allielas/Desktop/Active Projects/Quality control/Incucyte sheets/r1_processed.xlsx
Wrote replaced data to: /Users/allielas/Desktop/Active Projects/Quality control/Incucyte sheets/r2_processed.xlsx
Wrote replaced data to: /Users/allielas/Desktop/Active Projects/Quality control/Incucyte sheets/r3_processed.xlsx


/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  

Wrote replaced data to: /Users/allielas/Desktop/Active Projects/Quality control/Incucyte sheets/r4_processed.xlsx
Wrote replaced data to: /Users/allielas/Desktop/Active Projects/Quality control/Incucyte sheets/r5_processed.xlsx
Wrote replaced data to: /Users/allielas/Desktop/Active Projects/Quality control/Incucyte sheets/r6_processed.xlsx
Wrote replaced data to: /Users/allielas/Desktop/Active Projects/Quality control/Incucyte sheets/r7_processed.xlsx


/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/opt/homebrew/Caskroom/miniforge/base/envs/plotting/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [95]:
def load_mappings(path: Union[str, Path]):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Mappings file not found: {path}")
    else:
        # assume CSV/TSV: first two columns are old,new
        df = pd.read_csv(p, dtype=str)
        if df.shape[1] < 2:
            raise ValueError("CSV mappings must have at least two columns (old, new).")
        old_col, new_col = df.columns[0], df.columns[1]
        return {
            str(o): str(n)
            for o, n in zip(df[old_col].fillna(""), df[new_col].fillna(""))
        }


def parse_inline_maps(map_args: List[str]):
    mappings = {}
    for s in map_args:
        if ":" not in s:
            raise ValueError(f"Inline mapping must be OLD:NEW, got: {s!r}")
        old, new = s.split(":", 1)
        mappings[old] = new
    return mappings


def apply_replacements(df: pd.DataFrame, mapping: Dict[str, str]):
    if not mapping:
        return df.copy()
    # For performance: compile mapping items once
    items = list(mapping.items())

    def replace_value(x):
        """
        Docstring for replace_value. Just a simple string replace

        :str x: the string to replace based on the dict
        """
        if isinstance(x, str):
            for old, new in items:
                if old:
                    x = x.replace(old, new)
            return x
        return x

    return df.applymap(replace_value)